        # Scenario Bundle Tutorial 01 - Publish a scenario bundle using the API

        <!-- keep img below title and without align="left"  -->
<img src="https://raw.githubusercontent.com/OpenEnergyPlatform/organisation/refs/heads/production/logo/OpenEnergyFamily_Logo_OpenEnergyPlatform.svg" alt="OpenEnergy Platform" height="75" width="75" />

> **Verified on 2026-10-06 against OEP release v1.10.1.** The outputs
> below are a recorded run of this notebook against the live platform. If
> you see something different when you run it yourself, the API may have
> moved on since: the [API reference](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/api-reference/#/Scenario%20Bundles) is generated from the
> code and is always current, and the feedback issue below is the place to
> tell us.

## Learnings

A scenario bundle is the record of a study in the Open Energy Knowledge
Graph (OEKG): its scenarios, the publications it was written up in, and the
data those scenarios used and produced. The
[scenario bundle course](https://openenergyplatform.github.io/academy/courses/10_scenario_bundle/)
shows how to create one in the browser. This tutorial does the same from
Python, which is what you want once the information already sits in your
modelling pipeline.

This tutorial will enable you to:

- [Remove a bundle left over from an earlier run](#remove-a-leftover-from-an-earlier-run)
- [Create a scenario bundle](#create-a-scenario-bundle)
- [Read it back](#read-it-back)
- [Change one field, safely](#change-one-field-safely)
- [Add a scenario and cite the data it used](#add-a-scenario-and-cite-the-data-it-used)
- [Read what the writes recorded](#read-the-history)
- [Delete the bundle again](#delete-the-bundle)

The API behaves according to nine rules that no endpoint signature can
state. This tutorial links them where they matter instead of repeating
them, because they can change and this page is a snapshot: read them on
[Writing scenario bundles](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/).

## Requirements

- have your OEP API token at hand. You find it in the settings tab of your
  profile page. (If you don't have a token see
  [Things you only need to do once](https://openenergyplatform.github.io/academy/tutorials/99_other/beginners_guide/#1-things-you-only-need-to-do-once).)
- your OEP user name. The tutorial names its bundle after you, see
  [setup](#setup).
- Python
- Python packages:
  - `requests` (See the packages [documentation](https://requests.readthedocs.io/en/latest/) for more help)
  - jupyter notebook, e.g. `notebook` or `jupyterlab`
- Skills:
  - use [python](https://wiki.python.org/moin/BeginnersGuide) with [jupyter notebooks](https://docs.jupyter.org/en/latest/)
  - _some_ understanding of [JSON encoded data](https://www.w3schools.com/js/js_json_datatypes.asp)
- Internet: obviously, but also make sure your network settings allow https traffic from python

## What this tutorial writes

The knowledge graph has **no sandbox**. Everything this tutorial creates is
written to the public OEKG on the Open Energy Platform and is visible to
everyone until it is deleted. The tutorial therefore deletes its bundle at
the end, and also at the start, in case an earlier run was interrupted
before it got there. If you stop half-way, run the
[first section](#remove-a-leftover-from-an-earlier-run) again to clean up.

## Feedback

You can provide feedback on this tutorial in this [GitHub issue](https://github.com/OpenEnergyPlatform/academy/issues/TBD)

## Setup

This part of the code imports the required packages and asks for your OEP
API token and your OEP user name. Both are read from the environment
variables `OEP_API_TOKEN` and `OEP_USERNAME` if you have set them, and
asked for otherwise. Setting them keeps your token out of the notebook.

The user name decides the acronym of the bundle this tutorial creates,
`OEP-TUTORIAL-<your user name>`. The acronym is how the tutorial finds a
bundle it left behind, so it has to be yours: with a shared one, two
readers would delete each other's bundles. That is why the cell stops if
you give no user name rather than falling back to some default.

The two small functions at the end print what a response says and stop
the notebook when a request did not get the answer it expected.

In [ ]:
# install required packages with: pip install requests
# import required packages
import json
from getpass import getpass
from os import environ

import requests as req

OEP_URL = environ.get("OEP_URL", "https://openenergyplatform.org")
BUNDLES_URL = f"{OEP_URL}/api/v0/scenario-bundles/"

token = environ.get("OEP_API_TOKEN") or getpass("Enter your OEP API token:")
username = environ.get("OEP_USERNAME") or input("Enter your OEP user name:")
if not username.strip():
    raise ValueError(
        "This tutorial needs your OEP user name: it names its bundle after "
        "you, so that it can never delete somebody else's."
    )
ACRONYM = f"OEP-TUTORIAL-{username.strip()}"

# for writing, we need to add the authorization header
auth_headers = {"Authorization": "Token %s" % token}


def check(res, expected_status):
    """Print status and ETag of a response; stop if the status is unexpected."""
    print(res.status_code, res.reason, "| ETag:", res.headers.get("ETag"))
    if res.status_code != expected_status:
        raise Exception(res.text)


def show(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))


print("The tutorial bundle's acronym is", ACRONYM)

## Remove a leftover from an earlier run

Acronyms are unique on the platform, so if an earlier run of this tutorial
stopped before it deleted its bundle, creating the bundle again would be
refused. This cell looks for a bundle with the tutorial's acronym and
deletes it, so that every run starts from the same clean state.

Finding a bundle by its acronym needs no token, because reading is public.
The answer carries both things a delete needs: the bundle's identifier
(`_meta.uid`) and its current version (`_meta.version`). How the delete
itself works is explained [at the end](#delete-the-bundle).

This cell only ever deletes a bundle carrying exactly your tutorial
acronym, and the platform only lets you delete bundles you own.

In [ ]:
res = req.get(BUNDLES_URL, params={"acronym": ACRONYM})
check(res, 200)
leftovers = res.json()["results"]

for leftover in leftovers:
    res = req.delete(
        f"{BUNDLES_URL}{leftover['_meta']['uid']}/",
        params={"confirm": ACRONYM},
        headers={**auth_headers, "If-Match": f'"{leftover["_meta"]["version"]}"'},
    )
    check(res, 200)

print(f"Removed {len(leftovers)} leftover bundle(s) named {ACRONYM}.")

## Create a scenario bundle

This part of the code creates the bundle. A bundle needs a label and an
acronym, and the OEKG shape asks for more: an abstract, and at least one
descriptor, sector division, sector and technology. Those four are terms of
the [Open Energy Ontology](https://openenergyplatform.org/ontology/), sent
as their IRIs. The terms a field accepts are listed in the
[API reference](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/api-reference/#/Scenario%20Bundles/scenario_bundles_create).

A create is checked strictly: everything it would write is new, so any
missing piece is refused with a `400` that names it
([rule 6](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-6-a-write-is-judged-by-the-violations-it-introduces)).

The server mints the bundle's identifier
([rule 8](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-8-the-server-mints-the-identifier)). It comes back in the
`Location` header and in `_meta.uid`. The `ETag` header is the bundle's
version, which the next write has to send back.

In [ ]:
OEO = "https://openenergyplatform.org/ontology/oeo/"

bundle = {
    "label": "OEP Academy tutorial bundle",
    "acronym": ACRONYM,
    "abstract": (
        "Written by the OEP Academy tutorial on the scenario bundle API. "
        "The tutorial deletes it again when it finishes."
    ),
    "descriptors": [OEO + "OEO_00140133"],  # renewable energy share
    "sector_divisions": [OEO + "OEO_00010056"],  # KSG sector division
    "sectors": [OEO + "OEO_00010065"],  # KSG sector energy industry
    "technologies": [
        OEO + "OEO_00010424",  # wind power technology
        OEO + "OEO_00010428",  # photovoltaic technology
    ],
}

# Creating a bundle needs authentication headers
res = req.post(BUNDLES_URL, json=bundle, headers=auth_headers)
check(res, 201)

uid = res.json()["_meta"]["uid"]
etag = res.headers["ETag"]
bundle_url = f"{BUNDLES_URL}{uid}/"

print("Location:", res.headers["Location"])
print("You can see the bundle on the platform here:")
print(f"{OEP_URL}/scenario-bundles/id/{uid}")

## Read it back

Reading a bundle needs no token. `?expand=labels` adds the human-readable
labels of the ontology terms under `_meta.labels`.

Everything under `_meta` is the server's: the identifier, the IRI, the
version, the labels. A write that sends `_meta` back is not refused, it is
simply ignored ([rule 2](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-2-_meta-is-read-only-and-ignored-on-write)).

In [ ]:
res = req.get(bundle_url, params={"expand": "labels"})
check(res, 200)
show(res.json())

## Change one field, safely

A `PATCH` changes the fields it names and nothing else
([rule 1](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-1-a-key-you-do-not-send-is-a-key-you-do-not-change)), so you
never have to send the whole bundle to change its abstract.

Every write to a bundle that already exists has to say which version it
was prepared from, in an `If-Match` header carrying the `ETag` of the last
response. A write without one is refused with `428`.

In [ ]:
change = {"abstract": "Written by the OEP Academy tutorial, and since changed once."}

res = req.patch(bundle_url, json=change, headers=auth_headers)
check(res, 428)
show(res.json())

With the version from the create, the same request goes through. The
answer carries the bundle's new version, so the next write can chain on
without reading the bundle again.

In [ ]:
res = req.patch(bundle_url, json=change, headers={**auth_headers, "If-Match": etag})
check(res, 200)

old_etag, etag = etag, res.headers["ETag"]
print("abstract:", res.json()["abstract"])

This is what the header protects you from. Sending the old version again
is what a second script, or a colleague, would do if they had read the
bundle before your change: the write is refused with `412` instead of
silently undoing your change, and nothing is written.

In [ ]:
res = req.patch(
    bundle_url,
    json={"abstract": "An edit prepared from an outdated read."},
    headers={**auth_headers, "If-Match": old_etag},
)
check(res, 412)
show(res.json())

## Add a scenario and cite the data it used

Scenarios are not fields of the bundle but resources of their own, with
their own identifier and their own URL below the bundle. A `PATCH` on the
bundle cannot add one, a `POST` to its `scenarios/` can
([rule 4](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-4-a-bundle-post-accepts-nested-sub-resources-a-patch-does-not)).

The version is still the **bundle's**: every write below a bundle moves
the one version, and needs it in `If-Match`.

In [ ]:
scenario = {
    "label": "A high renewables scenario",
    "acronym": "HIGH-RE",
    "abstract": "Wind and solar power reach 80% of electricity generation.",
    "scenario_types": [OEO + "OEO_00020247"],  # target driven scenario
    "years": ["2030-01-01T00:00:00Z"],
}

res = req.post(
    f"{bundle_url}scenarios/",
    json=scenario,
    headers={**auth_headers, "If-Match": etag},
)
check(res, 201)

scenario_uid = res.json()["_meta"]["uid"]
etag = res.headers["ETag"]
show(res.json())

A scenario cites the data it used (`input`) and produced (`output`) with a
dataset link. `ref: "table"` names one table on the OEP and is the
reproducible citation: it always names that table. `ref: "dataset"` names a
dataset on the OEP, which groups tables, and always resolves to what the
dataset holds today.

A link is not checked when it is written, and it does not stop anyone from
deleting the table it names. Instead, every read says in `_meta` what the
link points at now: `resolvable` and the `tables` it currently resolves to
([rule 7](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-7-ref-table-is-reproducible-ref-dataset-is-current-and-a-dead-link-stays)).

The example cites `tutorial_example_table`, the table the
[API tutorials](https://openenergyplatform.github.io/academy/tutorials/01_api/02_api_upload/)
work with.

In [ ]:
link = {"type": "input", "ref": "table", "name": "tutorial_example_table"}

res = req.post(
    f"{bundle_url}scenarios/{scenario_uid}/datasets/",
    json=link,
    headers={**auth_headers, "If-Match": etag},
)
check(res, 201)

etag = res.headers["ETag"]
show(res.json())

### Why this tutorial adds no study report

A bundle also lists the study reports, the publications its study was
written up in, at its own `study-reports/`
([reference](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/api-reference/#/Scenario%20Bundles/scenario_bundles_study_reports_create)). This
tutorial leaves them out on purpose. A study report needs at least one
author, and authors, like contacts and organisations, are shared between
all bundles. Deleting a bundle only detaches it from them and never deletes
them ([rule 5](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-5-the-deletes-two-guards-catch-two-different-accidents)),
so a demonstration report would leave an author behind in the public
knowledge graph on every run.

In your own bundle, add your study reports: for an author who is already
in the graph, send their `iri` along with their exact `label`; for a new
one, send only the `label`.

## Read the history

Every write is recorded in the bundle's history: who made it, when, which
version it turned into which, and what it changed. The history is public
and newest first. The list below summarises each entry: the request, the
resource it was about, the versions either side of it, who made it and
which fields it set. The full entries also list every value that was
added and removed.

In [ ]:
# the OEO classes of what a history entry can be about
KINDS = {
    OEO + "OEO_00020227": "bundle",
    OEO + "OEO_00000365": "scenario",
    OEO + "OEO_00030029": "dataset link",
}

res = req.get(f"{bundle_url}history/")
check(res, 200)

for entry in res.json()["results"]:
    fields = sorted({c["field"] for c in entry["changes"] or [] if c["field"]})
    print(
        entry["timestamp"][:19],
        entry["verb"].ljust(7),
        KINDS.get(entry["resource"]["type"], "other").ljust(12),
        f"version {entry['version_before']} -> {entry['version_after']}",
        "by",
        entry["actor"],
        "|",
        ", ".join(fields) or "-",
    )

## Delete the bundle

Deleting a whole bundle is irreversible, so it takes two pieces of
evidence that you mean it, and each catches a different mistake
([rule 5](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/#rule-5-the-deletes-two-guards-catch-two-different-accidents)):

- `If-Match` with the current version catches **stale state**: the bundle
  changed since you looked at it.
- `?confirm=<acronym>` catches the **wrong bundle**: the identifier in the
  URL is not the bundle you meant. The acronym has to match exactly.

That is why a delete starts with a read: it gives you the version and the
acronym. Without the confirmation the delete is refused, and nothing is
deleted.

In [ ]:
res = req.get(bundle_url)
check(res, 200)
etag = res.headers["ETag"]
acronym = res.json()["acronym"]

res = req.delete(bundle_url, headers={**auth_headers, "If-Match": etag})
check(res, 400)
show(res.json())

With both, the bundle is deleted. The answer lists what the delete
removed (`deleted`): the bundle, its scenario and its dataset link. A
part that another bundle also uses would be detached instead of deleted,
and listed under `unlinked`.

In [ ]:
res = req.delete(
    bundle_url,
    params={"confirm": acronym},
    headers={**auth_headers, "If-Match": etag},
)
check(res, 200)
show(res.json())

The bundle is gone: reading it answers `404`. Its history is not. It keeps
one line saying who deleted the bundle, when, and under which acronym.

In [ ]:
res = req.get(bundle_url)
check(res, 404)

res = req.get(f"{bundle_url}history/")
check(res, 200)
latest = res.json()["results"][0]
print(latest["timestamp"][:19], latest["verb"], latest["actor"], latest["acronym"])

## Now do it for real

You have created, changed, extended and deleted a scenario bundle, and the
knowledge graph is as it was before you started. Now describe your own
study the same way, with your own acronym, your scenarios, your study
reports and the tables your scenarios used and produced, and leave out the
final delete.

- [Writing scenario bundles](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/oekg-api/scenario-bundles/): the nine rules that decide what a
  request does. Read rule 3 before you automate anything.
- [API reference](https://openenergyplatform.github.io/oeplatform/oeplatform-code/web-api/api-reference/#/Scenario%20Bundles): every endpoint, header and status code.
- [Scenario Bundle Tutorial 02](https://openenergyplatform.github.io/academy/tutorials/05_scenario_bundles/02_scenario_bundle_pipeline/): if
  your pipeline should update its bundle on every run.
- [Scenario bundle course](https://openenergyplatform.github.io/academy/courses/10_scenario_bundle/):
  the same in the browser.

## About this tutorial

:oep-logo-nfdi4energy:

- Author: Jonas Huber
- Copyright: [Reiner Lemoine Institut](https://reiner-lemoine-institut.de) (2026)
- License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.en)
- Attribution: Reiner Lemoine Institut (2026): Scenario Bundle Tutorial 01 - Publish a scenario bundle using the API.
- Last update :oep-auto-lastupdate: